# Konu 5 — Dil modelleri nasıl çalışır?

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Isınma ve Adım 1–5 **anahtarsız** çalışır. Adım 6–7 `anahtar.txt` ister.

## Isınma — cümleyi tamamla

Sınıfa sorduk: *"Sessiz bir çalışma ..."* cümlesinin sonunu ne getirirsin?
Altı kişinin cevabı aşağıda. En sık iki cevap hangisi? `Counter` Konu 1'den tanıdık.

In [ ]:
from collections import Counter

cevaplar = ["kafesi", "kafesi", "yeri", "odası", "kafesi", "yeri"]
sayac = Counter(cevaplar)
print(sayac.most_common(2))

Çıktıdaki her parantez bir **ikili**: kelime ve adedi. Konu 1'de bu ikilileri `for kelime, adet in ...` ile almıştık.

Bir sonraki kelimeyi tahmin etmenin en basit yolu: **en sık söyleneni seç.**
Bu konunun bütün fikri bu hücrede. Dil modeli de bunu yapar, yalnızca milyarlarca metinle.

## Adım 1 — Model metni nasıl görür: belirteçler

**Tahmin et:** *"Sessiz bir çalışma kafesi"* dört kelime; model kaç parça görecek? Tahminini sohbete yaz.

Sonra tarayıcıda: `tiktokenizer.vercel.app` sitesine aynı cümleyi yaz, renkli parçaları say.
Model metni **kelime kelime** değil, **parça parça** okur. Bu parçalara **belirteç** (token) denir.

Şimdi aynısını kodla. `tiktoken` bu konunun yeni kütüphanesi (`uv sync` ile kuruldu). `get_encoding` hazır bir **parça sözlüğü** yükler:
modelin tanıdığı bütün parçaların numaralı listesi (Python'daki `{}` sözlüğüyle karıştırma). `"o200k_base"` bu listenin adı.
`kodlayici.encode` metni parçalara ayırıp her parçayı listedeki numarasına çevirir.

İlk çalıştırmada parça sözlüğü internetten iner; birkaç saniye sürebilir.
İnternet yoksa `ConnectionError` alırsın: Adım 2'den devam et (Adım 2–5 parça sözlüğünü kullanmaz), Adım 1'e internet gelince dön.

Kod, sitede saydığın sayıyı vermeli:

In [ ]:
import tiktoken

kodlayici = tiktoken.get_encoding("o200k_base")
belirtecler = kodlayici.encode("Sessiz bir çalışma kafesi")
print(len(belirtecler))
print(belirtecler)

Model kelimeleri değil, bu **sayıları** görür. Her sayı bir parça. Sayıları yeniden parçalara çevirelim.
`decode` bir **liste** ister; o yüzden tek parçayı köşeli parantez içinde veriyoruz:

In [ ]:
for belirtec in belirtecler:
    print(kodlayici.decode([belirtec]))

Dört kelime, altı parça. `bir` ve `çalışma` tek parça, ama `Sessiz` ve `kafesi` ikiye bölündü.
Parçanın başındaki boşluk da parçaya dahil.

Aynı anlamda İngilizce bir cümle kaç parça?

In [ ]:
ingilizce = kodlayici.encode("A quiet cafe for working")
print(len(ingilizce))

Beş kelime, beş parça. Türkçe, ekleri yüzünden daha çok parçaya bölünür. Uzun bir Türkçe kelime deneyelim.
**Tahmin et:** iki kelime; kaç parça?

In [ ]:
uzun = kodlayici.encode("Kütüphanedekilerden misiniz?")
print(len(uzun))
for belirtec in uzun:
    print(kodlayici.decode([belirtec]))

İki kelime, on bir parça. Model için Türkçe metin daha "uzun": aynı sözü söylemek için daha çok
parça harcar. Modelin bir seferde okuyabildiği parça sayısı sınırlıdır (**bağlam penceresi**; Konu 2 Adım 8'de
soruya koyduğumuz bağlam da bu pencereye girer). Türkçe metin bu pencereyi daha çabuk doldurur.
Ücretli modellerde fiyat da parça başına: aynı broşür metnini Türkçe yazdırmak daha pahalı.

## Adım 2 — Hangi kelimeden sonra ne geliyor?

Isınmadaki soruya dönüyoruz: bir kelimeden sonra ne gelir? Model bunu parçalarla yapar; biz kolay okunsun diye **kelimelerle** yapacağız.
Konu 1'deki 30 kafe yorumunu okuyup Konu 1'deki satırlarla temizliyoruz (dosyanın kopyası bu konunun `veri` klasöründe).
Çıktıda tek bir sayı görmelisin: yorumlarda toplam kaç kelime var.

In [ ]:
with open("veri/kafe-yorumlari.txt", encoding="utf-8") as dosya:
    metin = dosya.read()

temiz = metin.replace("I", "ı")
temiz = temiz.replace("İ", "i")
temiz = temiz.lower()
for isaret in ".,;:!?'()-":
    temiz = temiz.replace(isaret, " ")

kelimeler = temiz.split()
print(len(kelimeler))

Şimdi "biraz" kelimesinden **hemen sonra** gelen kelimeleri topluyoruz.
Döngü her kelimede bir önceki kelimeye bakar: önceki "biraz" ise bu kelimeyi listeye ekler.
Son satır, bu kelimeyi bir sonraki tur için "önceki" olarak hatırlar.

In [ ]:
hedef = "biraz"
sonra_gelenler = []
onceki = ""
for kelime in kelimeler:
    if onceki == hedef:
        sonra_gelenler.append(kelime)
    onceki = kelime

print(sonra_gelenler)

Müşteriler "biraz"dan sonra ne yazmış? Dördü de ne tür kelimeler?
Dosyada hangi yorumlardan geldiklerine bak. Kafeyi yeniden tasarlayan biri için bu bir iş listesi:
hangisi tasarımcının işi, hangisi değil?

Şimdi "çok" için aynısını istiyoruz. Satırları yeniden yazmak yerine bir **fonksiyona** koyalım
(Konu 2'deki `modele_sor` gibi): kelimeyi alsın, ondan sonra gelenlerin listesini geri versin.
Bu hücre bir şey yazmaz; fonksiyonu yalnızca tanımlar. Bir sonraki hücre onu çağırır.

In [ ]:
def sonra_gelenleri_bul(hedef):
    sonra_gelenler = []
    onceki = ""
    for kelime in kelimeler:
        if onceki == hedef:
            sonra_gelenler.append(kelime)
        onceki = kelime
    return sonra_gelenler

In [ ]:
sonra_gelenler = sonra_gelenleri_bul("çok")
print(sonra_gelenler)

Hangisi en sık? Isınmadaki `Counter`:

In [ ]:
sayac = Counter(sonra_gelenler)
print(sayac.most_common(5))

"biraz" şikâyetleri getirmişti; "çok" ne getirdi? Dosyada "çok" geçen yorumlara bak: beşinden üçü bahçeyle ilgili.
"biraz" neyi **düzelteceğini**, "çok" neyi **koruyacağını** söylüyor.

## Adım 3 — Sayıdan olasılığa

"çok"tan sonra 5 kelime gelmiş; "güzel" bunların 2'si. Yani "güzel"in **olasılığı** 2 / 5.
Olasılık "kaçta kaç" demek: 0 ile 1 arası bir oran (Konu 3'teki doğruluk gibi).
Her kelimenin adedini toplama bölüyoruz:

In [ ]:
toplam = len(sonra_gelenler)
etiketler = []
olasiliklar = []
for kelime, adet in sayac.most_common(5):
    olasilik = adet / toplam
    etiketler.append(kelime)
    olasiliklar.append(olasilik)
    print(kelime, olasilik)

Olasılıkları topla: kaç ediyor? Şimdi bunları Konu 1 Adım 8'deki gibi bir çubuk grafikle çizelim:

In [ ]:
import matplotlib.pyplot as plt

plt.bar(etiketler, olasiliklar, color="#4a6fa5")
plt.title("'çok' kelimesinden sonra ne geliyor?")
plt.ylabel("Olasılık")
plt.show()

Bu grafiğe **olasılık dağılımı** denir: bütün seçeneklerin olasılıkları bir arada.
Bir dil modeli de her adımda böyle bir dağılım hesaplar: "bundan sonra hangi parça, hangi olasılıkla?"
Farkı, bunu 30 yorumdan değil milyarlarca metinden öğrenmiş olması.

## Adım 4 — Metin üret: hep en olasıyı seç

Yeni fonksiyon bir kelime alır, ondan sonra **en sık** gelen kelimeyi geri verir.
Adım 2'deki fonksiyondan listeyi alır, `Counter` ile sayar.
`most_common(1)` yalnızca en sık bir kelimeyi verir, yine bir liste içinde: `[('güzel', 2)]`.
`en_sik[0]` bir ikili: kelime ve adedi; ikisini iki değişkene alıyoruz.

In [ ]:
def sonraki_kelime(hedef):
    sonra_gelenler = sonra_gelenleri_bul(hedef)
    sayac = Counter(sonra_gelenler)
    en_sik = sayac.most_common(1)
    kelime, adet = en_sik[0]
    return kelime

**Tahmin et:** "çok" için ne yazacak? İpucu: Adım 3'teki grafiğin en uzun çubuğu.

In [ ]:
print(sonraki_kelime("çok"))

Metin üretmek = bu tahmini tekrarlamak. "kahve"den başlıyoruz; her turda son kelimenin ardından
en olası kelimeyi ekliyoruz. Listede 8 sayı var: döngü 8 tur döner.
**Tahmin et:** anlamlı bir cümle çıkar mı?

In [ ]:
kelime = "kahve"
cumle = kelime
for tur in [1, 2, 3, 4, 5, 6, 7, 8]:
    kelime = sonraki_kelime(kelime)
    cumle = cumle + " " + kelime

print(cumle)

Yan yana duran her kelime çifti gerçek bir yorumda geçiyor; ama bütün cümle bir şey anlatıyor mu?
Hücreyi bir kez daha çalıştır: değişti mi?

Hep en olasıyı seçince her seferinde **aynı** cümle çıkar. Model bir anlam kurmuyor; yalnızca sayıyor.

Cümle bir müşteri yorumuna benziyor ama hiçbir müşteri bunu yazmadı. Bir kafe sitesinde "yorum" diye dursa fark eder miydin?

## Adım 5 — Zar at

Şimdi en sıkını değil, **zarla** seçelim. `random` Python'la birlikte gelir, kurulum istemez; `random.choice` listeden rastgele bir eleman seçer.
"çok"tan sonra gelenler listesinde "güzel" iki kez var; o yüzden seçilme şansı da iki kat.
Hücreyi birkaç kez çalıştır:

In [ ]:
import random

print(sonra_gelenler)
print(random.choice(sonra_gelenler))

Yeni fonksiyon da Adım 4'teki gibi listeyi alır; ama en sıkını değil, zarla seçtiğini verir.
Dosyanın son kelimesi "kahveli"den sonra hiçbir kelime gelmiyor. Zar oraya gelirse liste boş kalır:
metin bittiyse **baştan başla**, metnin ilk kelimesini ver.
`len(...) == 0` "liste boş mu?" demek. `return` fonksiyonu orada bitirir; liste boş değilse son satıra geçilir.

In [ ]:
def sonraki_kelime_zarla(hedef):
    sonra_gelenler = sonra_gelenleri_bul(hedef)
    if len(sonra_gelenler) == 0:
        return kelimeler[0]
    return random.choice(sonra_gelenler)

Aynı üretim, zarla. Hücreyi üç kez çalıştır, cümleleri karşılaştır:

In [ ]:
kelime = "kahve"
cumle = kelime
for tur in [1, 2, 3, 4, 5, 6, 7, 8]:
    kelime = sonraki_kelime_zarla(kelime)
    cumle = cumle + " " + kelime

print(cumle)

Her çalıştırmada başka bir cümle. Arada "kahveli ders" görürsen: zar metnin sonuna geldi,
üretim baştan ("ders çalışmak için...") devam etti.

Gerçek modellerde bu ayarın adı **sıcaklık** (Konu 4'teki haritanın "sıcaklık" ekseniyle ilgisi yok; buradaki sıcaklık zarın ayarı):

- Sıcaklık 0'a yakın ≈ hep en olasıyı seç (Adım 4): her seferinde aynı
- Sıcaklık 1 civarı ≈ olasılığa göre zar at (Adım 5): her seferinde farklı
- Sıcaklık 1'in üstü: zar hileli; olasılığı düşük kelimeler daha çok şans bulur

## Adım 6 — Gerçek model ve sıcaklık

Adım 2–5'te 30 yorumdan sayan **kendi küçük modelimizi** kurduk. Gerçek model de her adımda sıradaki
parçayı seçer; farkı milyarlarca metinden öğrenmiş olması. En olasıyı mı seçeceğini, zar mı atacağını
biz **sıcaklıkla** ayarlayacağız.

Buradan sonrası `anahtar.txt` ister ve önceki hücrelere bağlı değildir: yeni bir derse buradan başlayabilirsin.
Konu 2'nin Adım 1 hücresinin aynısı; `Anahtar okundu` görmelisin:

In [ ]:
anahtarlar = {}
with open("../anahtar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        parcalar = satir.split("=")
        ad = parcalar[0].strip()
        deger = parcalar[1].strip()
        anahtarlar[ad] = deger

hesap = anahtarlar["ACCOUNT_ID"]
anahtar = anahtarlar["API_TOKEN"]
print("Anahtar okundu")

Adres ve başlık Konu 2'deki gibi. `modele_sor` fonksiyonu da Konu 2'deki; tek fark sıcaklık:
fonksiyon onu ikinci bilgi olarak alır (`sicaklik`) ve gövdedeki `"temperature"` alanına koyar.

In [ ]:
import requests

MODEL = "@cf/meta/llama-3.3-70b-instruct-fp8-fast"
adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{MODEL}"
basliklar = {"Authorization": f"Bearer {anahtar}"}

In [ ]:
def modele_sor(soru, sicaklik):
    govde = {"prompt": soru, "temperature": sicaklik}
    cevap = requests.post(adres, headers=basliklar, json=govde)
    yanit = cevap.json()
    sonuc = yanit["result"]
    return sonuc["response"]

Aynı soruyu üç sıcaklıkta soruyoruz.
**Çalıştırmadan önce tahmin et:** hangisi en "garip" slogan olacak?

In [ ]:
soru = "Sessiz bir çalışma kafesi için tek bir kısa slogan yaz. Yalnızca sloganı yaz."

for sicaklik in [0.1, 0.7, 1.5]:
    metin = modele_sor(soru, sicaklik)
    print(sicaklik, metin)

0.1'deki slogan başka bir kafenin tabelasında da durabilir mi? Adım 4'ü hatırla: hep en olası = hep en sık söylenen.

Düşük sıcaklıkta iki kez soralım. İki cevap aynı mı?

In [ ]:
ilk = modele_sor(soru, 0.1)
ikinci = modele_sor(soru, 0.1)
print(ilk)
print(ikinci)

Şimdi yüksek sıcaklıkta iki kez:

In [ ]:
ilk = modele_sor(soru, 1.5)
ikinci = modele_sor(soru, 1.5)
print(ilk)
print(ikinci)

Adım 4 ile Adım 5'i hatırla: hangisi hangisine benziyor?

## Adım 7 — Slogan panosu

Bu kez modelden tek seferde **üç** slogan istiyoruz. Her sıcaklığın sloganlarını başlığıyla
tek bir dosyaya yazıyoruz. `"w"` "yazmak için aç", `"\n"` yeni satır. Modele 3 istek gider.
`str(sicaklik)` sayıyı metne çevirir (Konu 3'teki `int`'in tersi): metne yalnızca metin eklenebilir.

In [ ]:
pano_sorusu = "Sessiz bir çalışma kafesi için üç kısa slogan yaz. Her slogan ayrı satırda olsun. Yalnızca sloganları yaz."

with open("pano.txt", "w", encoding="utf-8") as dosya:
    for sicaklik in [0.1, 0.7, 1.5]:
        metin = modele_sor(pano_sorusu, sicaklik)
        dosya.write("Sıcaklık " + str(sicaklik) + "\n")
        dosya.write(metin + "\n")
        dosya.write("\n")

print("pano.txt yazıldı")

Klasörde `pano.txt`'yi aç, üç bölümü yan yana oku.

- Kurumsal bir tabela için hangi sıcaklık?
- Fikir fırtınası için hangisi?
- 1.5'teki sloganlardan kullanılabilecek olan var mı?
- Önce fikir toplayıp sonra son metni yazdıracak olsan hangi sırayla hangi sıcaklıkları kullanırdın?

## Bonus

**1.** Kendi adını ve soyadını belirteçlere ayır (Adım 1'deki `kodlayici`; derse Adım 6'dan başladıysan önce Adım 1'in ilk hücresini çalıştır). Kaç parça? Yanındakinin adı kaç parça? Kafe için aklına gelen bir ad?

In [ ]:
belirtecler = kodlayici.encode("Ahmet Emre Aladağ")
print(len(belirtecler))
for belirtec in belirtecler:
    print(kodlayici.decode([belirtec]))

**2.** Adım 4'teki üretim hücresinde `"kahve"` yerine `"sessiz"` ya da `"priz"` yaz, yeniden çalıştır.
Dikkat: dosyanın son kelimesi `"kahveli"` ile başlarsan `IndexError` alırsın. Ondan sonra hiç kelime gelmediği için
`most_common(1)` boş liste verir; `en_sik[0]` diye bir eleman yoktur.

**3.** (Anahtar ister.) Adım 6'daki `soru` satırına Adım 2'nin bir bulgusunu ekle, ör. *"bahçesi olan sessiz bir çalışma kafesi"*.
Adım 6'nın hücrelerini yeniden çalıştır. Slogan bahçeden söz ediyor mu? (Konu 2 Adım 8: model senin bulgunu, sen yazmazsan bilmez.)